# Chatbot Legal VietNam — Data Generation (v2)

**Cải tiến so với v1:**
-  Prompt cấm từ chỉ thị mơ hồ: 'này', 'đó', 'nêu trên', 'trên đây', 'dưới đây'
-  Bắt buộc dùng tên văn bản cụ thể (van_ban) + tên điều khoản (Điều X, Khoản Y)
-  Thêm few-shot examples TỐT/XẤU động theo metadata
-  Filter `AMBIGUOUS_PATTERNS` loại query mơ hồ còn sót
-  Retry tự động nếu generate ra từ mơ hồ hoặc ký tự Hán
-  Monitor `skipped_amb` trong vòng lặp
-  CELL 13 audit chất lượng cuối cùng

## 1. Setup

In [1]:
# CELL 1 — Cài thư viện cần thiết
!python -m pip -q install -U "transformers>=4.41.0" "accelerate>=0.30.0" "bitsandbytes>=0.46.1" "sentencepiece"

In [2]:
# CELL 2 — Kiểm tra GPU / phiên bản
import torch, sys
print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
CUDA available: True
GPU: Tesla T4
Torch: 2.10.0+cu128 CUDA: 12.8


In [4]:
!rm -rf /content/drive

In [5]:
# CELL 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Load dữ liệu

In [6]:
# CELL 4 — Load chunks
import json

CHUNKS_PATH = "/content/drive/MyDrive/Legal chat bot/Data/chunks.json"
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Total chunks:", len(chunks))
print("Chunk keys  :", list(chunks[0].keys()))
print("Meta keys   :", list((chunks[0].get("metadata") or {}).keys()))

# Kiểm tra nhanh 1 chunk mẫu
sample = chunks[0]
md = sample.get("metadata", {}) or {}
print("\n--- Sample chunk ---")
print("van_ban :", md.get("van_ban"))
print("dieu    :", md.get("dieu"))
print("text[:200]:", sample.get("text", "")[:200])

Total chunks: 1861
Chunk keys  : ['text', 'metadata']
Meta keys   : ['van_ban', 'chuong', 'dieu', 'khoan', 'diem', 'source_file']

--- Sample chunk ---
van_ban : NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Tư pháp
dieu    : 1
text[:200]: Theo Điều 1 NGHỊ ĐỊNH Quy định về phân định thẩm quyền của chính quyền địa phương 02 cấp trong lĩnh vực quản lý nhà nước của Bộ Tư pháp Phạm vi điều chỉnh Nghị định này quy định về việc phân định thẩm


## 3. Load model

In [7]:
# CELL 5 — Load Qwen2.5-7B-Instruct (4-bit quantized)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.config.use_cache = False  # tiết kiệm VRAM khi generate

print("Model loaded. Device:", next(model.parameters()).device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded. Device: cuda:0


## 4. Prompt & Helpers

> **Thay đổi chính ở CELL 6:** Prompt v2 bắt buộc LLM dùng tên văn bản cụ thể, cấm từ mơ hồ, có few-shot examples.

In [8]:
# CELL 6 — SYSTEM_MSG + build_prompt (v2: chống từ mơ hồ)
import json

# ════════════════════════════════════════════════════════════════
# SYSTEM_MSG — 3 lớp ràng buộc:
#   1. CHỈ tiếng Việt, cấm ký tự CJK Hán
#   2. CẤM từ chỉ thị mơ hồ: 'này', 'đó', 'nêu trên', ...
#   3. BẮT BUỘC dùng TÊN VĂN BẢN + TÊN ĐIỀU KHOẢN cụ thể
# ════════════════════════════════════════════════════════════════
SYSTEM_MSG = (
    "Bạn là công cụ sinh câu hỏi pháp lý tiếng Việt để huấn luyện hệ thống RAG. "
    "Mỗi câu hỏi PHẢI tự đứng vững (self-contained): người đọc chỉ nhìn câu hỏi "
    "mà không có ngữ cảnh vẫn biết đang hỏi về văn bản / điều khoản nào. "
    "BẮT BUỘC: (1) Chỉ dùng tiếng Việt có dấu. "
    "(2) TUYỆT ĐỐI KHÔNG dùng từ chỉ thị mơ hồ: "
    "'này', 'đó', 'kia', 'trên', 'dưới', 'nêu trên', 'như trên', 'đã nêu', "
    "'trên đây', 'dưới đây', 'sau đây', 'được đề cập'. "
    "(3) PHẢI dùng TÊN VĂN BẢN và TÊN ĐIỀU KHOẢN CỤ THỂ từ thông tin đã cung cấp. "
    "(4) KHÔNG dùng ký tự Hán, tiếng Trung, hay tiếng Anh. "
    "Không giải thích. Không markdown. Trả về DUY NHẤT JSON."
)


def build_prompt(chunk: dict, n_questions: int = 4) -> str:
    md  = chunk.get("metadata", {}) or {}
    doc = md.get("van_ban", "Văn bản pháp luật")

    chuong = md.get("chuong")
    dieu   = md.get("dieu")
    khoan  = md.get("khoan")
    diem   = md.get("diem")

    ref_parts = []
    if chuong: ref_parts.append(f"Chương {chuong}")
    if dieu:   ref_parts.append(f"Điều {dieu}")
    if khoan:  ref_parts.append(f"Khoản {khoan}")
    if diem:   ref_parts.append(f"Điểm {diem}")
    ref = " - ".join(ref_parts) if ref_parts else "Phần chung"

    # ── Few-shot examples động theo metadata ──────────────────
    ex_doc   = f"'{doc}'"
    ex_dieu  = f"Điều {dieu} của {ex_doc}" if dieu  else ex_doc
    ex_khoan = f"Khoản {khoan}, {ex_dieu}" if khoan else ex_dieu

    good1 = f"  ✅ {ex_doc} quy định điều kiện nào để được cấp phép?"
    good2 = f"  ✅ {ex_dieu} quy định về thời hạn nộp hồ sơ như thế nào?"
    good3 = f"  ✅ Chủ thể nào có trách nhiệm thực hiện quy định tại {ex_khoan}?"

    passage = chunk.get("text", "")

    prompt = (
        f"Văn bản : {doc}\n"
        f"Tham chiếu: {ref}\n"
        f"\n"
        f"Đoạn trích:\n"
        f'"""{passage}"""\n'
        f"\n"
        f"Yêu cầu:\n"
        f"- Sinh {n_questions} câu hỏi tiếng Việt tự nhiên mà người dùng thực tế có thể hỏi.\n"
        f"- Câu hỏi phải trả lời được CHỈ dựa trên đoạn trích.\n"
        f"- TUYỆT ĐỐI KHÔNG dùng từ mơ hồ: 'này', 'đó', 'kia', 'nêu trên', 'trên đây', 'dưới đây'.\n"
        f"- PHẢI dùng tên văn bản {ex_doc} hoặc tên điều khoản ({ref}) khi đề cập đến văn bản / điều khoản.\n"
        f"- Không nhắc 'đoạn trích', 'đoạn văn', 'dựa vào đoạn'.\n"
        f"- Không thêm giải thích, không thêm markdown.\n"
        f"- TUYỆT ĐỐI KHÔNG dùng tiếng Trung / English hoặc ký tự Hán.\n"
        f"\n"
        f"Ví dụ câu hỏi TỐT (làm theo pattern này):\n"
        f"{good1}\n"
        f"{good2}\n"
        f"{good3}\n"
        f"\n"
        f"Ví dụ câu hỏi XẤU (KHÔNG bao giờ làm theo):\n"
        f"  ❌ 'Nghị định này quy định điều kiện nào?'\n"
        f"  ❌ 'Điều này quy định gì về thời hạn?'\n"
        f"  ❌ 'Quy định nêu trên áp dụng cho ai?'\n"
        f"\n"
        f'Trả về DUY NHẤT JSON đúng schema:\n'
        f'{{"queries": ["...", "...]}}'
    )
    return prompt

In [9]:
# CELL 7 — safe_parse_queries (robust: 3 cách parse)
# FIX: thêm tham số max_q thay vì dùng biến n_questions ngoài scope
import json, re

def safe_parse_queries(output_text: str, max_q: int = 4) -> list:
    s = output_text.strip()

    # Cách 1: parse trực tiếp
    try:
        data = json.loads(s)
        qs = data.get("queries", [])
        if isinstance(qs, list):
            return [q.strip() for q in qs if isinstance(q, str) and len(q.strip()) >= 10]
    except Exception:
        pass

    # Cách 2: tìm JSON object cuối cùng trong văn bản
    last_r = output_text.rfind("}")
    if last_r != -1:
        for start in range(output_text.rfind("{", 0, last_r), -1, -1):
            if output_text[start] != "{":
                continue
            blob = output_text[start:last_r + 1]
            try:
                data = json.loads(blob)
                qs = data.get("queries", [])
                if isinstance(qs, list):
                    return [q.strip() for q in qs
                            if isinstance(q, str) and len(q.strip()) >= 10]
            except Exception:
                continue

    # Cách 3: fallback tách dòng (lấy câu hỏi kết thúc bằng ?)
    lines = [ln.strip().strip('",') for ln in output_text.splitlines()]
    candidates = [ln for ln in lines if len(ln) >= 10 and ln.endswith("?")]
    return candidates[:max_q] if candidates else []

In [10]:
# CELL 8 — is_valid_query: lọc rác + ký tự Hán + từ chỉ thị mơ hồ
import re

# ── 1. Query rác / leakage từ prompt ──────────────────────────
BAD_PATTERNS = [
    r"\bbạn là\b",
    r"\bdựa vào\b",
    r"\bđoạn trích\b",
    r"\bđoạn văn\b",
    r"\bhãy tạo\b",
    r"\bsinh câu hỏi\b",
    r"\btheo đoạn\b",
    r"\bví dụ\b",
]

# ── 2. Từ chỉ thị mơ hồ ───────────────────────────────────────
# Lý do loại: retriever không biết 'nghị định này' là văn bản
# nào → embedding lấy sai tài liệu → recall giảm.
AMBIGUOUS_PATTERNS = [
    # "X này"
    r"\bnghị định này\b",
    r"\bquy định này\b",
    r"\bluật này\b",
    r"\bthông tư này\b",
    r"\bquyết định này\b",
    r"\bvăn bản này\b",
    r"\bchỉ thị này\b",
    r"\bpháp lệnh này\b",
    r"\bnghị quyết này\b",
    r"\bđiều này\b",
    r"\bkhoản này\b",
    r"\bđiểm này\b",
    r"\bchương này\b",
    r"\bmục này\b",
    # "X đó"
    r"\bnghị định đó\b",
    r"\bquy định đó\b",
    r"\bluật đó\b",
    r"\bvăn bản đó\b",
    # cụm vị trí
    r"\bnêu trên\b",
    r"\bnhư trên\b",
    r"\bđã nêu\b",
    r"\btrên đây\b",
    r"\bdưới đây\b",
    r"\bsau đây\b",
    r"\bđược đề cập\b",
]

# ── 3. Ký tự Hán ──────────────────────────────────────────────
HAN_RE = re.compile(r"[\u3400-\u4DBF\u4E00-\u9FFF]")

# Compile 1 lần để tăng tốc
_bad_re = [re.compile(p) for p in BAD_PATTERNS]
_amb_re = [re.compile(p) for p in AMBIGUOUS_PATTERNS]


def is_valid_query(q: str) -> bool:
    q = q.strip()
    if len(q) < 12:
        return False
    if HAN_RE.search(q):
        return False
    low = q.lower()
    for pat in _amb_re:
        if pat.search(low):
            return False
    for pat in _bad_re:
        if pat.search(low):
            return False
    return True

In [11]:
# CELL 9 — generate_queries (retry khi xuất hiện Hán HOẶC từ mơ hồ)
import torch, re

HAN_RE_GEN = re.compile(r"[\u3400-\u4DBF\u4E00-\u9FFF]")

# Detect từ mơ hồ trong raw output (trước khi parse JSON)
AMB_QUICK_RE = re.compile(
    r"\b(nghị định này|quy định này|luật này|thông tư này|"
    r"văn bản này|điều này|khoản này|nêu trên|như trên|"
    r"trên đây|dưới đây|sau đây|đã nêu|được đề cập)\b",
    re.IGNORECASE,
)


def _contains_han(text: str) -> bool:
    return bool(HAN_RE_GEN.search(text or ""))


def _contains_ambiguous(text: str) -> bool:
    return bool(AMB_QUICK_RE.search(text or ""))


@torch.no_grad()
def generate_queries(chunk: dict, n_questions: int = 4,
                     max_new_tokens: int = 300,
                     temperature: float = 0.25,
                     max_retries: int = 3) -> str:
    user_prompt = build_prompt(chunk, n_questions=n_questions)

    def _gen_once(sys_msg: str, temp: float) -> str:
        messages = [
            {"role": "system", "content": sys_msg},
            {"role": "user",   "content": user_prompt},
        ]
        enc = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )
        if isinstance(enc, torch.Tensor):
            input_ids    = enc.to(model.device)
            attention_mask = None
            prompt_len   = input_ids.shape[-1]
        else:
            input_ids      = enc["input_ids"].to(model.device)
            attention_mask = enc.get("attention_mask")
            if attention_mask is not None:
                attention_mask = attention_mask.to(model.device)
            prompt_len = input_ids.shape[-1]

        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temp,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
        gen_ids = out[0][prompt_len:]
        return tokenizer.decode(gen_ids, skip_special_tokens=True)

    # Lần 1: generate bình thường
    out_text = _gen_once(SYSTEM_MSG, temperature)

    # Retry nếu output lỗi
    for attempt in range(1, max_retries + 1):
        has_han = _contains_han(out_text)
        has_amb = _contains_ambiguous(out_text)
        if not has_han and not has_amb:
            break  # ✅ output sạch

        reasons = []
        if has_han: reasons.append("CÓ KÝ TỰ HÁN")
        if has_amb: reasons.append("CÓ TỪ MƠ HỒ ('này','đó','nêu trên',...)")

        strict_msg = (
            SYSTEM_MSG
            + f" VI PHẠM ({'; '.join(reasons)}) Ở LẦN TRƯỚC."
            + " Sinh lại: CHỈ tiếng Việt, KHÔNG từ mơ hồ,"
            + " PHẢI dùng tên văn bản cụ thể. Trả về DUY NHẤT JSON."
        )
        out_text = _gen_once(strict_msg, max(0.05, temperature - 0.07 * attempt))

    return out_text

In [12]:
# CELL 10 — clean_gpu
import gc, torch

def clean_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Sinh dataset

In [13]:
# CELL 11 — Sinh dataset JSONL (có RESUME + monitor mơ hồ)
import os, json, time, random

OUT_DIR    = "/content/drive/MyDrive/Legal chat bot/Data"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(OUT_DIR, "train.jsonl")
DEV_PATH   = os.path.join(OUT_DIR, "dev.jsonl")

# ── Cấu hình ────────────────────────────────────────────────────
NQ         = 4      # số câu hỏi mỗi chunk
MAX_NEW    = 300    # max_new_tokens (tăng để chứa few-shot JSON)
SAVE_EVERY = 25     # log & checkpoint mỗi N chunk
DEV_RATIO  = 0.1   # 10% → dev set

# ── Helpers ─────────────────────────────────────────────────────
def write_jsonl(path, rows, mode="a"):
    with open(path, mode, encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def count_lines(path):
    if not os.path.exists(path):
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

def get_last_chunk_index(path):
    """Đọc vài KB cuối file để lấy chunk_index lớn nhất (tránh load toàn bộ)."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return -1
    with open(path, "rb") as f:
        f.seek(0, os.SEEK_END)
        size = f.tell()
        f.seek(max(0, size - 8192))
        data = f.read()
    for line in reversed(data.splitlines()):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line.decode("utf-8"))
            idx = (obj.get("meta") or {}).get("chunk_index")
            if isinstance(idx, int):
                return idx
        except Exception:
            continue
    return -1

# ── RESUME ──────────────────────────────────────────────────────
last_done = max(get_last_chunk_index(TRAIN_PATH),
                get_last_chunk_index(DEV_PATH))
start_i   = max(1, last_done + 2)   # i là 1-based

print(f"Resume từ chunk i = {start_i} (1-based)")
print(f"Train lines: {count_lines(TRAIN_PATH)}, Dev lines: {count_lines(DEV_PATH)}")

# ── Vòng lặp sinh ───────────────────────────────────────────────
total       = len(chunks)
t0          = time.time()
skipped_amb = 0    # monitor số query bị lọc do mơ hồ

for i in range(start_i, total + 1):
    chunk = chunks[i - 1]
    print(f"[{i}/{total}] generating...")

    output_text = generate_queries(chunk, n_questions=NQ, max_new_tokens=MAX_NEW)
    queries     = safe_parse_queries(output_text, max_q=NQ)  # truyền NQ vào

    if not queries:
        clean_gpu()
        continue

    md   = chunk.get("metadata", {}) or {}
    rows = []
    for q in queries:
        if not is_valid_query(q):
            skipped_amb += 1
            continue
        rows.append({
            "query"  : q,
            "passage": chunk.get("text", ""),
            "label"  : 1,
            "meta"   : {
                "van_ban"    : md.get("van_ban"),
                "chuong"     : md.get("chuong"),
                "dieu"       : md.get("dieu"),
                "khoan"      : md.get("khoan"),
                "diem"       : md.get("diem"),
                "source_file": md.get("source_file"),
                "chunk_index": i - 1,
            },
        })

    if rows:
        for r in rows:
            if random.random() < DEV_RATIO:
                write_jsonl(DEV_PATH,   [r], mode="a")
            else:
                write_jsonl(TRAIN_PATH, [r], mode="a")

    clean_gpu()

    if i % SAVE_EVERY == 0:
        elapsed = time.time() - t0
        print(f"  ↳ Checkpoint {i}/{total} | {elapsed:.1f}s | skipped_ambiguous={skipped_amb}")

print("\nDONE.")
print(f"Train : {TRAIN_PATH}  ({count_lines(TRAIN_PATH)} dòng)")
print(f"Dev   : {DEV_PATH}  ({count_lines(DEV_PATH)} dòng)")
print(f"Query bị lọc do mơ hồ: {skipped_amb}")

Resume từ chunk i = 1806 (1-based)
Train lines: 3915, Dev lines: 429
[1806/1861] generating...
[1807/1861] generating...
[1808/1861] generating...
[1809/1861] generating...
[1810/1861] generating...
[1811/1861] generating...
[1812/1861] generating...
[1813/1861] generating...
[1814/1861] generating...
[1815/1861] generating...
[1816/1861] generating...
[1817/1861] generating...
[1818/1861] generating...
[1819/1861] generating...
[1820/1861] generating...
[1821/1861] generating...
[1822/1861] generating...
[1823/1861] generating...
[1824/1861] generating...
[1825/1861] generating...
  ↳ Checkpoint 1825/1861 | 563.3s | skipped_ambiguous=0
[1826/1861] generating...
[1827/1861] generating...
[1828/1861] generating...
[1829/1861] generating...
[1830/1861] generating...
[1831/1861] generating...
[1832/1861] generating...
[1833/1861] generating...
[1834/1861] generating...
[1835/1861] generating...
[1836/1861] generating...
[1837/1861] generating...
[1838/1861] generating...
[1839/1861] gener

## 6. Thêm Hard Negatives

In [14]:
# CELL 12 — Thêm hard negatives (ưu tiên cùng van_ban)
import json, random, os

NEG_TRAIN_PATH = os.path.join(OUT_DIR, "train_with_neg.jsonl")

# Index chunks theo van_ban
by_doc = {}
for idx, ch in enumerate(chunks):
    doc = (ch.get("metadata") or {}).get("van_ban", "UNKNOWN")
    by_doc.setdefault(doc, []).append(idx)


def pick_negative(chunk_idx: int) -> str:
    ch  = chunks[chunk_idx]
    doc = (ch.get("metadata") or {}).get("van_ban", "UNKNOWN")
    candidates = by_doc.get(doc, [])
    if len(candidates) >= 2:
        # Hard negative: cùng văn bản nhưng khác chunk
        j = random.choice([x for x in candidates if x != chunk_idx])
    else:
        # Fallback: random toàn cục
        j = random.randrange(len(chunks))
        while j == chunk_idx:
            j = random.randrange(len(chunks))
    return chunks[j].get("text", "")


open(NEG_TRAIN_PATH, "w", encoding="utf-8").close()

with open(TRAIN_PATH, "r", encoding="utf-8") as f_in, \
     open(NEG_TRAIN_PATH, "a", encoding="utf-8") as f_out:
    for line in f_in:
        pos = json.loads(line)
        chunk_idx = (pos.get("meta") or {}).get("chunk_index")
        if chunk_idx is None:
            continue
        # Ghi positive
        f_out.write(json.dumps(pos, ensure_ascii=False) + "\n")
        # Ghi negative
        neg = dict(pos)
        neg["passage"] = pick_negative(chunk_idx)
        neg["label"]   = 0
        f_out.write(json.dumps(neg, ensure_ascii=False) + "\n")

print("Wrote:", NEG_TRAIN_PATH)

Wrote: /content/drive/MyDrive/Legal chat bot/Data/train_with_neg.jsonl


## 7. Audit chất lượng

Chạy sau khi sinh xong để kiểm tra tỉ lệ query mơ hồ còn sót.

In [15]:
# CELL 13 — Audit: đếm query mơ hồ còn sót
import json, re

AMB_AUDIT_RE = re.compile(
    r"\b(nghị định này|quy định này|luật này|thông tư này|"
    r"văn bản này|điều này|khoản này|mục này|chương này|"
    r"nêu trên|như trên|đã nêu|trên đây|dưới đây|sau đây|"
    r"được đề cập|quy định đó)\b",
    re.IGNORECASE,
)

for fname, label in [(TRAIN_PATH, "train"), (DEV_PATH, "dev")]:
    total, amb = 0, 0
    amb_examples = []
    with open(fname, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            total += 1
            q = d.get("query", "")
            if AMB_AUDIT_RE.search(q):
                amb += 1
                if len(amb_examples) < 3:
                    amb_examples.append(q)
    pct = amb / total * 100 if total else 0
    print(f"[{label}] ambiguous: {amb}/{total}  ({pct:.1f}%)")
    for ex in amb_examples:
        print(f"   ❌ {ex}")

print("\n✅ Mục tiêu: tỉ lệ mơ hồ < 5%")

[train] ambiguous: 0/3954  (0.0%)
[dev] ambiguous: 0/430  (0.0%)

✅ Mục tiêu: tỉ lệ mơ hồ < 5%
